# Student Depression Dataset.
Analyzing Mental Health Trends and Predictors Among Students

A student depression dataset typically contains data aimed at analyzing, understanding, and predicting depression levels among students. It may include features such as demographic information (age, gender), academic performance (grades, attendance), lifestyle habits (sleep patterns, exercise, social activities), mental health history, and responses to standardized depression scales.

These datasets are valuable for research in psychology, data science, and education to identify factors contributing to student depression and to design early intervention strategies. Ethical considerations like privacy, informed consent, and anonymization of data are crucial in working with such sensitive information.

Link = https://www.kaggle.com/datasets/hopesb/student-depression-dataset?resource=download

## READ THE DATA

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Import 9 Algoritma
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

## Read Data


In [2]:
df = pd.read_csv('data/Student Depression Dataset.csv')

In [3]:
df.head()

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,M.Tech,Yes,1.0,1.0,No,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27901 entries, 0 to 27900
Data columns (total 18 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   id                                     27901 non-null  int64  
 1   Gender                                 27901 non-null  str    
 2   Age                                    27901 non-null  float64
 3   City                                   27901 non-null  str    
 4   Profession                             27901 non-null  str    
 5   Academic Pressure                      27901 non-null  float64
 6   Work Pressure                          27901 non-null  float64
 7   CGPA                                   27901 non-null  float64
 8   Study Satisfaction                     27901 non-null  float64
 9   Job Satisfaction                       27901 non-null  float64
 10  Sleep Duration                         27901 non-null  str    
 11  Dietary Habit

## PREPROCESSING

In [5]:
df.isnull().sum()

id                                       0
Gender                                   0
Age                                      0
City                                     0
Profession                               0
Academic Pressure                        0
Work Pressure                            0
CGPA                                     0
Study Satisfaction                       0
Job Satisfaction                         0
Sleep Duration                           0
Dietary Habits                           0
Degree                                   0
Have you ever had suicidal thoughts ?    0
Work/Study Hours                         0
Financial Stress                         3
Family History of Mental Illness         0
Depression                               0
dtype: int64

In [6]:
df = df.dropna(subset=['Financial Stress'])

In [7]:
if 'id' in df.columns:
    df = df.drop('id', axis=1)

In [8]:
df = df.reset_index(drop=True)

In [9]:
kolom_le = ['Gender', 'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness']
kolom_ohe = ['City', 'Profession', 'Sleep Duration', 'Dietary Habits', 'Degree']

In [10]:
## Label Encoding
le = LabelEncoder()
for col in kolom_le:
    df[col] = le.fit_transform(df[col])

In [11]:
##One-Hot Encoding
ohe = OneHotEncoder(sparse_output=False, drop='first')
res_ohe = pd.DataFrame(ohe.fit_transform(df[kolom_ohe]), columns=ohe.get_feature_names_out(kolom_ohe))

In [12]:
df = pd.concat([df.drop(columns=kolom_ohe), res_ohe], axis=1)

In [13]:
print("Shape data:", df.shape)

Shape data: (27898, 110)


## TRAIN TEST SPLIT AND STANDAR SCALER

In [14]:
X = df.drop('Depression', axis=1)
y = df['Depression']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [16]:
scaler = StandardScaler()

In [17]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Model Training and Model Selection

In [18]:
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted, average='weighted', zero_division=0)
    recall = recall_score(true, predicted, average='weighted', zero_division=0)
    f1 = f1_score(true, predicted, average='weighted', zero_division=0)
    return accuracy, precision, recall, f1

In [19]:
## Beginning Model Training
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "SVM": SVC(),
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "XGBoost": XGBClassifier()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train model

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Evaluate Train and Test dataset
    train_acc, train_prec, train_rec, train_f1 = evaluate_model(y_train, y_train_pred)

    test_acc, test_prec, test_rec, test_f1 = evaluate_model(y_test, y_test_pred)

    
    print(list(models.keys())[i])
    
    print('Model performance for Training set')
    # print("- Accuracy:  {:.4f}".format(train_acc))
    # print("- Precision: {:.4f}".format(train_prec))
    # print("- Recall:    {:.4f}".format(train_rec))
    # print("- F1 Score:  {:.4f}".format(train_f1))

    print('----------------------------------')
    
    print('Model performance for Test set')
    # print("- Accuracy:  {:.4f}".format(test_acc))
    # print("- Precision: {:.4f}".format(test_prec))
    # print("- Recall:    {:.4f}".format(test_rec))
    print("- F1 Score:  {:.4f}".format(test_f1))
    
    print('='*35)
    print('\n')

Logistic Regression
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.8448


Decision Tree
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.7850


Random Forest
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.8374


SVM
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.8440


Naive Bayes
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.2529


KNN
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.8025


Gradient Boosting
Model performance for Training set
----------------------------------
Model performance for Test set
- F1 Score:  0.8470


AdaBoost
Model performance for Training set
---------

Best 3 Model
1. AdaBoost
2. Gradient Boosting
3. Logistic Regression

In [24]:
#Initialize few parameter for Hyperparamter tuning
adaboost_parms = {
    "n_estimators": [50, 100, 200, 500], 
    "learning_rate": [0.01, 0.1, 1.0, 1.5, 2.0] 
}
gradient_boost_parms = {
    "n_estimators": [100, 200],
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5, 8],
    "min_samples_split": [2, 5],
    "subsample": [0.8, 1.0]
}
logistic_regression_parms = {
    "C": [0.01, 0.1, 1, 10, 100],     
    "penalty": ['l1', 'l2'],           
    "solver": ['liblinear', 'saga'],   
    "max_iter": [500, 1000]            
}

In [25]:
gridcv_models = [
                   ("Adaboost",AdaBoostClassifier() ,adaboost_parms),
                   ("Gradient_Boost",GradientBoostingClassifier(),gradient_boost_parms),
                   ("Logistic Regression",LogisticRegression(),logistic_regression_parms)
                   
                   ]

In [26]:
## Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV

model_param = {}

for name, model, params in gridcv_models:
    grid = GridSearchCV(estimator=model,
                        param_grid=params, 
                        cv=3,
                        verbose=2,
                        n_jobs=-1)
    grid.fit(X_train, y_train)
    model_param[name] = grid.best_params_

for model_name in model_param:
    print(f"---------------- Best Params for {model_name} -------------------")
    print(model_param[model_name])


Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 72 candidates, totalling 216 fits
Fitting 3 folds for each of 40 candidates, totalling 120 fits
---------------- Best Params for Adaboost -------------------
{'learning_rate': 1.5, 'n_estimators': 200}
---------------- Best Params for Gradient_Boost -------------------
{'learning_rate': 0.2, 'max_depth': 3, 'min_samples_split': 5, 'n_estimators': 100, 'subsample': 0.8}
---------------- Best Params for Logistic Regression -------------------
{'C': 1, 'max_iter': 500, 'penalty': 'l2', 'solver': 'saga'}


In [27]:
from sklearn.base import clone

comparison_results = []

for name, model, params in gridcv_models:
    print(f"Evaluating {name}...")
    
    default_model = clone(model)
    default_model.fit(X_train, y_train)
    
    y_pred_default = default_model.predict(X_test)
    
    def_acc, def_prec, def_rec, def_f1 = evaluate_model(y_test, y_pred_default)
    
    best_params = model_param[name]
    
    tuned_model = clone(model)
    
    tuned_model.set_params(**best_params) 
    
    tuned_model.fit(X_train, y_train)
    y_pred_tuned = tuned_model.predict(X_test)
    
    tun_acc, tun_prec, tun_rec, tun_f1 = evaluate_model(y_test, y_pred_tuned)
    
    ## Calculate
    f1_improvement = tun_f1 - def_f1
    acc_improvement = tun_acc - def_acc
    
    comparison_results.append({
        'Model Name': name,
        'Default F1-Score': def_f1,
        'Tuned F1-Score': tun_f1,
        'F1 Improvement': f1_improvement,
        'Default Accuracy': def_acc,
        'Tuned Accuracy': tun_acc,
        'Acc Improvement': acc_improvement
    })

## Comparsion Tabel
df_comparison = pd.DataFrame(comparison_results)
df_comparison = df_comparison.sort_values(by='F1 Improvement', ascending=False).reset_index(drop=True)

import IPython.display as display
print("\n" + "="*80)
print("📊 COMPARISON TABLE: DEFAULT PARAMETERS vs HYPERPARAMETER TUNING RESULTS 📊")
print("="*80)
display.display(df_comparison)

print("\n--- HOW TO READ THE RESULTS ---")
print("1. Positive (+) Improvement: Tuning SUCCESSFULLY made the model better.")
print("2. Negative (-) Improvement: Tuning FAILED (likely overfitting on the selected parameters). Recommendation: Just use the Default model.")
print("3. Zero (0) Improvement: The Default parameter is already the most optimal setting.")


Evaluating Adaboost...
Evaluating Gradient_Boost...
Evaluating Logistic Regression...

📊 COMPARISON TABLE: DEFAULT PARAMETERS vs HYPERPARAMETER TUNING RESULTS 📊


,Model Name,Default F1-Score,Tuned F1-Score,F1 Improvement,Default Accuracy,Tuned Accuracy,Acc Improvement
0,Gradient_Boost,0.846968,0.849745,0.002777,0.848029,0.850717,0.002688
1,Logistic Regression,0.844816,0.845551,0.000735,0.845878,0.846595,0.000717
2,Adaboost,0.847071,0.845672,-0.001399,0.848029,0.846774,-0.001254



--- HOW TO READ THE RESULTS ---
1. Positive (+) Improvement: Tuning SUCCESSFULLY made the model better.
2. Negative (-) Improvement: Tuning FAILED (likely overfitting on the selected parameters). Recommendation: Just use the Default model.
3. Zero (0) Improvement: The Default parameter is already the most optimal setting.
